## TEST DATASET INITIALISATION

In [ ]:
import sys
sys.path.append('../')

import numpy as np
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data_utils

# import datasets
# import importlib
# importlib.reload(datasets)

from utils import dec2bin, dec2base, base2dec
from random_hierarchy_model import sample_rules, sample_trees, sample_trees_unif, RHM
from mixed_random_hierarchy_model import MixedRandomHierarchyModel_varying_tree

In [ ]:
from dataclasses import dataclass
@dataclass
class config:

    device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

    num_features = 16   # The number of values each variable can take (vocabulary size, int).
    num_classes = 16   # The number of classes (int).
    binary_rules = 1   # The number of synonymic lower-level representations (multiplicity, int).
    num_layers = 4     # The number of levels in the hierarchy (int).

    seed_rules = 12345678
    #seed_sample = 56781234
    seed_sample=11 
    train_size = 2**20
    test_size = 0

print(config.device)

In [ ]:
rhm = MixedRandomHierarchyModel_varying_tree(
    num_features=config.num_features,
    num_classes=config.num_classes,
    binary_rules=config.binary_rules,
    num_layers=config.num_layers,
    seed_rules=config.seed_rules,
    seed_sample=config.seed_sample,
    train_size=config.train_size,
    test_size=config.test_size,
    input_format="long",
    transform=None
)
rhm.features

In [ ]:
batch =rhm.features
pad_token = config.num_features

In [ ]:
import json
import numpy as np

rules_array = {}
for key in rhm.rules.keys():
    rules_array[f"rule{key+1}"] = rhm.rules[key].numpy()

np.savez(f"mixed_rhm.L{config.num_layers}v{config.num_features}bin{config.binary_rules}.{config.seed_rules}.rules.npz", **rules_array)


In [ ]:
data_dict = {
    'vocab_size': config.num_features,
    'L': config.num_layers,
    'pad_token_id': int(pad_token),
    'd_max': int(batch.shape[1]),
    'seed_rules': config.seed_rules,
    'train_size': config.train_size,
    'test_size': config.test_size,
    'seed_sample': config.seed_sample,
}

with open(f"mixed_rhm.L{config.num_layers}v{config.num_features}bin{config.binary_rules}.{config.seed_rules}.meta.json", "w") as f:
    json.dump(data_dict, f, indent=2)


In [ ]:
# save padded sentence array — shape [N, d_max], pad_token = num_features
np.save(
    f"mixed_rhm.L{config.num_layers}v{config.num_features}bin{config.binary_rules}.{config.seed_rules}.npy",
    batch.numpy().astype(np.int32)
)


In [ ]:
# save train split — shape [N_train, d_max]
np.save(
    f"mixed_rhm.L{config.num_layers}v{config.num_features}bin{config.binary_rules}.{config.seed_rules}.e22.train.npy",
    batch.numpy().astype(np.int32)
)


In [ ]:
# save test split — shape [N_test, d_max]
np.save(
    f"mixed_rhm.L{config.num_layers}v{config.num_features}bin{config.binary_rules}.{config.seed_rules}.e20.test.npy",
    batch.numpy().astype(np.int32)
)


## Generating data from given rules

In [ ]:
# here I generate random rules, but they could be taken from a file or another simulation
seed_rules = random.randint(10000000,99999999)
rules = sample_rules( config.num_features, config.num_classes, config.num_synonyms, config.tuple_size, config.num_layers, seed=seed_rules)

In [ ]:
dataset_size = 1024
seed_samples = random.randint(10000000,99999999)

trees = sample_trees(dataset_size, rules, prior=None, probs=None, seed=seed_samples)
inputs = trees[config.num_layers]
targets = trees[0]